# Importar librerías

In [1]:
import os
import time
import arcpy

# Manipulación raster

## Descarga y procesamiento por lotes

### Definición de funciones

In [76]:
def nodata_para_clip(ruta):
    r = arcpy.Raster(ruta)
    nd = r.noDataValue
    print(f"{ruta} | tipo píxel: {r.pixelType} | bandas: {r.bandCount} | NoData: {nd}")
    return None if nd is None else str(nd)

def descargarTeselas(raster, grilla, gdb_intermedia, nombre):

    t1 = time.time()
    
    sr_raster = arcpy.Describe(raster).spatialReference
    print(f"Spatial reference raster: {sr_raster.name}")
    sr_grilla = arcpy.Describe(grilla).spatialReference
    print(f"Spatial reference grilla: {sr_raster.name}")
    
    nodata = nodata_para_clip(raster)
    opciones = {"nodata_value": nodata} if nodata is not None else {}
    
    with arcpy.EnvManager(snapRaster=raster):
        with arcpy.da.SearchCursor(grilla, ["OID@", "SHAPE@"]) as cursor:
            for oid, geom in cursor:
    
                # Si la rejilla está en otro CRS que el raster, se proyecta la celda
                if sr_grilla.factoryCode != sr_raster.factoryCode:
                    geom = geom.projectAs(sr_raster)
    
                extension = geom.extent
                print(f"Extensión tesela {oid}: {extension}")
                salida = f"{gdb_intermedia}\\{nombre}_t{oid:02d}"
                print(f"Archivo de salida: {salida}")
    
                # Si la tesela ya existe, se salta (útil si el servidor se cae a la mitad)
                if arcpy.Exists(salida):
                    print(f"Tesela {oid}: ya existe, se omite")
                    continue
    
                print(f"Tesela {oid}: pidiendo al servidor...")
                arcpy.management.Clip(
                    in_raster=raster,
                    rectangle=extension,
                    out_raster=salida,
                    clipping_geometry="NONE",
                    maintain_clipping_extent="NO_MAINTAIN_EXTENT",
                    **opciones
                )
                print(f"Tesela {oid}: lista")
                print("\n---------------------------------------------------------------------------\n")
                
    tiempo = time.time() - t1
    
    print(f"Teselas {nombre} descargadas exitosamente en {tiempo:.0f} segundos")

def componerMosaico(gdb_intermedia, nombre, gdb_salida, nombre_mosaico, n_teselas=32):

    t1 = time.time()
    
    # Lista las teselas por patrón de nombre
    arcpy.env.workspace = gdb_intermedia
    teselas = sorted(arcpy.ListRasters(f"{nombre}_t*"))
    print(f"Teselas encontradas: {len(teselas)}")
    if len(teselas) != n_teselas:
        raise RuntimeError(f"Se esperaban {n_teselas} teselas. Revisa cuáles faltan antes de continuar.")
    
    # Traducción de los códigos de arcpy.Raster a los de MosaicToNewRaster
    TIPOS = {
        "U1": "1_BIT", "U2": "2_BIT", "U4": "4_BIT",
        "U8": "8_BIT_UNSIGNED", "S8": "8_BIT_SIGNED",
        "U16": "16_BIT_UNSIGNED", "S16": "16_BIT_SIGNED",
        "U32": "32_BIT_UNSIGNED", "S32": "32_BIT_SIGNED",
        "F32": "32_BIT_FLOAT", "F64": "64_BIT",
    }
    
    # Propiedades de la primera tesela, y comprobación de que todas coinciden
    ref = arcpy.Raster(teselas[0])
    for t in teselas:
        r = arcpy.Raster(t)
        if (r.pixelType, r.bandCount) != (ref.pixelType, ref.bandCount):
            raise RuntimeError(f"{t} difiere de la primera tesela: {r.pixelType}, {r.bandCount} bandas")
    
    if ref.pixelType not in TIPOS:
        raise RuntimeError(f"Tipo de píxel sin traducción: {ref.pixelType}")
    
    print(f"Celda: {ref.meanCellWidth} | bandas: {ref.bandCount} | tipo: {ref.pixelType} -> {TIPOS[ref.pixelType]}")
    
    arcpy.management.MosaicToNewRaster(
        input_rasters                      = teselas,
        output_location                    = gdb_salida,
        raster_dataset_name_with_extension = nombre_mosaico,
        coordinate_system_for_the_raster   = ref.spatialReference,
        pixel_type                         = TIPOS[ref.pixelType],      # el mismo de las teselas
        cellsize                           = ref.meanCellWidth,
        number_of_bands                    = ref.bandCount,
        mosaic_method                      = "FIRST",
        mosaic_colormap_mode               = "FIRST"
    )
    
    tiempo = time.time() - t1
    
    print(f"Mosaico {nombre} creado exitosamente en {tiempo:.0f} segundos")

def validarMosaico(gdb_intermedia, nombre, gdb_salida, nombre_mosaico, borrar=False):

    # Lista las teselas por patrón de nombre
    arcpy.env.workspace = gdb_intermedia
    teselas = sorted(arcpy.ListRasters(f"{nombre}_t*"))
    ref = arcpy.Raster(teselas[0])

    # Ubica el mosaico
    mosaico = f"{gdb_salida}\\{nombre_mosaico}"
    m = arcpy.Raster(mosaico)
    
    # Extensión combinada de las teselas
    xmin = min(arcpy.Raster(t).extent.XMin for t in teselas)
    ymin = min(arcpy.Raster(t).extent.YMin for t in teselas)
    xmax = max(arcpy.Raster(t).extent.XMax for t in teselas)
    ymax = max(arcpy.Raster(t).extent.YMax for t in teselas)
    tol = ref.meanCellWidth * 2
    
    checks = {
        "bandas":    m.bandCount == ref.bandCount,
        "tipo":      m.pixelType == ref.pixelType,
        "celda":     abs(m.meanCellWidth - ref.meanCellWidth) < 1e-6,
        "CRS":       m.spatialReference.factoryCode == ref.spatialReference.factoryCode,
        "extension": (abs(m.extent.XMin - xmin) < tol and abs(m.extent.YMin - ymin) < tol and
                      abs(m.extent.XMax - xmax) < tol and abs(m.extent.YMax - ymax) < tol),
    }
    for k, v in checks.items():
        print(f"{k}: {'OK' if v else 'FALLA'}")
    
    if all(checks.values()) and borrar:
        for t in teselas:
            arcpy.management.Delete(f"{gdb_intermedia}\\{t}")
        print(f"Teselas {nombre} eliminadas exitosamente")
    else:
        print(f"No se borraron las teselas {nombre}")

def procesarCapaRaster(raster, grilla, gdb_intermedia, gdb_salida, sufijo, n_teselas=32, borrar=False):

    t1 = time.time()
    
    # Definición de variables
    nombre = os.path.basename(raster)
    nombre_mosaico = f"{nombre}_{sufijo}"
    
    print(f"La capa {nombre} se está procesando")

    # Procesamiento
    #descargarTeselas(raster, grilla, gdb_intermedia, nombre)
    #componerMosaico(gdb_intermedia, nombre, gdb_salida, nombre_mosaico, n_teselas)
    validarMosaico(gdb_intermedia, nombre, gdb_salida, nombre_mosaico, borrar)
    
    tiempo = time.time() - t1
    
    print(f"La capa {nombre} fue procesada exitosamente en {tiempo:.0f} segundos")
    print("\n---------------------------------------------------------------------------\n")

### Ejecutar proceso

In [ ]:
from pathlib import Path

# Definición de capas a procesar. Corresponden a los servicios REST
# de las ortofotos cargados en el mapa de ArcGIS Pro.
capas = [
    r"ServiciosImagen\Ortofoto_Medellin_2016_Centro",
    r"ServiciosImagen\Ortofoto_Medellin_2019",
    r"ServiciosImagen\Ortofoto_Medellin_2021_MAGNA_Medellin",
    r"ServiciosImagen\Ortofoto_Medellin_2024",
    r"ServiciosImagen\Modelo_digital_de_elevacion_medellin_2024",
    r"ServiciosImagen\Ortofotos_2004_SIGMA"
]

# Argumentos de función.
# Reemplazar el placeholder por la ruta local del directorio del proyecto.
PROJECT_DIR = Path("<RUTA_LOCAL_DEL_PROYECTO>")

grilla         = str(PROJECT_DIR / "urban_creek.gdb" / "aoi_calasanz_teselas_mini")
gdb_intermedia = str(PROJECT_DIR / "intermediate.gdb")
gdb_salida     = str(PROJECT_DIR / "raw.gdb")
sufijo         = "calasanz"
n_teselas      = 32
borrar         = True

# Ejecución de la función de procesamiento iterativa.
for capa in capas:
    raster = capa
    procesarCapaRaster(
        raster, grilla, gdb_intermedia, gdb_salida, sufijo, n_teselas, borrar
    )

La capa Ortofoto_Medellin_2016_Centro se está procesando
bandas: OK
tipo: OK
celda: OK
CRS: OK
extension: OK
Teselas Ortofoto_Medellin_2016_Centro eliminadas exitosamente
La capa Ortofoto_Medellin_2016_Centro fue procesada exitosamente en 44 segundos

---------------------------------------------------------------------------

La capa Ortofoto_Medellin_2019 se está procesando
bandas: OK
tipo: OK
celda: OK
CRS: OK
extension: OK
Teselas Ortofoto_Medellin_2019 eliminadas exitosamente
La capa Ortofoto_Medellin_2019 fue procesada exitosamente en 48 segundos

---------------------------------------------------------------------------

La capa Ortofoto_Medellin_2021_MAGNA_Medellin se está procesando
bandas: OK
tipo: OK
celda: OK
CRS: OK
extension: OK
Teselas Ortofoto_Medellin_2021_MAGNA_Medellin eliminadas exitosamente
La capa Ortofoto_Medellin_2021_MAGNA_Medellin fue procesada exitosamente en 67 segundos

---------------------------------------------------------------------------

La capa Mo

## Validación rásteres

### Validación preliminar

In [42]:
arcpy.env.workspace = r"D:\urban_creek\01_operations\urban_creek\raw.gdb"

print(f"{'CAPA':<30}{'|':<2}{'CRS':<5}{'|':<2}{'CLSZ':<5} | {'NBAND':<5} | {'TYPE':<5} | {'COMP':<5}")
print(f"-------------------------------------------------------------------")
for n in arcpy.ListRasters():
   r = arcpy.Raster(n)
   print(f"{n:<30}{'|':<2}{r.spatialReference.factoryCode:<5}{'|':<2}{r.meanCellWidth:<5.2f} | {r.bandCount:<5} | {r.pixelType:<5} | {r.compressionType:<5}")

CAPA                          | CRS  | CLSZ  | NBAND | TYPE  | COMP 
-------------------------------------------------------------------
ortofoto_2010_calasanz        | 6257 | 0.15  | 3     | U8    | LZ77 
ortofoto_2016_calasanz        | 6257 | 0.07  | 3     | U8    | LZ77 
ortofoto_2019_calasanz        | 6257 | 0.08  | 3     | U8    | LZ77 
ortofoto_2021_calasanz        | 6257 | 0.08  | 3     | U8    | LZ77 
DTM_2024_calasanz             | 9377 | 1.00  | 1     | F32   | LZ77 
ortofoto_2004_calasanz        | 6257 | 0.13  | 3     | U8    | LZ77 
ortofoto_2024_calasanz        | 9377 | 0.08  | 3     | U8    | LZ77 


### Reconstrucción de pirámides y estadísticas ráster

In [ ]:
for year in (2004, 2010, 2016, 2019, 2021, 2024):

    # Patrón de nombramiento de las capas de ortofotos dentro del proyecto de ArcGIS Pro.
    capa = f"Ortofotos\ortofoto_{year}_calasanz"
    
    print(f"Destruyendo pirámides para {capa}")
    arcpy.management.BuildPyramids(
        in_raster_dataset=capa,
        pyramid_level=0,
        SKIP_FIRST="NONE",
        resample_technique="NEAREST",
        compression_type="DEFAULT",
        compression_quality=75,
        skip_existing="OVERWRITE"
    )

    print(f"Reconstruyendo pirámides para {capa}")
    arcpy.management.BuildPyramids(
        in_raster_dataset=capa,
        pyramid_level=-1,
        SKIP_FIRST="NONE",
        resample_technique="BILINEAR",
        compression_type="LZ77",
        compression_quality=75,
        skip_existing="OVERWRITE"
    )

    print(f"Calculando estadísticas para {capa}")
    arcpy.management.CalculateStatistics(
        in_raster_dataset=capa,
        x_skip_factor=1,
        y_skip_factor=1,
        ignore_values=[],
        skip_existing="OVERWRITE",
        area_of_interest=r"in_memory\feature_set1"
    )
    print(f"Proceso completado exitosamente para para {capa}")
    

<>:3: SyntaxWarning: invalid escape sequence '\o'
C:/Users/USER/AppData/Local/Temp/ArcGISProTemp2440/xpython_2440/3439458572.py:3: SyntaxWarning: invalid escape sequence '\o'
  capa = f"Ortofotos\ortofoto_{year}_calasanz"


Destruyendo pirámides para Ortofotos\ortofoto_2004_calasanz
Reconstruyendo pirámides para Ortofotos\ortofoto_2004_calasanz
Proceso completado exitosamente para para Ortofotos\ortofoto_2004_calasanz
Destruyendo pirámides para Ortofotos\ortofoto_2010_calasanz
Reconstruyendo pirámides para Ortofotos\ortofoto_2010_calasanz
Proceso completado exitosamente para para Ortofotos\ortofoto_2010_calasanz
Destruyendo pirámides para Ortofotos\ortofoto_2016_calasanz
Reconstruyendo pirámides para Ortofotos\ortofoto_2016_calasanz
Proceso completado exitosamente para para Ortofotos\ortofoto_2016_calasanz
Destruyendo pirámides para Ortofotos\ortofoto_2019_calasanz
Reconstruyendo pirámides para Ortofotos\ortofoto_2019_calasanz
Proceso completado exitosamente para para Ortofotos\ortofoto_2019_calasanz
Destruyendo pirámides para Ortofotos\ortofoto_2021_calasanz
Reconstruyendo pirámides para Ortofotos\ortofoto_2021_calasanz
Proceso completado exitosamente para para Ortofotos\ortofoto_2021_calasanz
Destruyend

### Copiado por lotes

In [ ]:
out_gdb = gdb_intermedia

for year in (2010, 2016, 2019):

    capa = f"Ortofotos\ortofoto_{year}_calasanz"

    print(f"Copiando ráster {capa}")
    arcpy.management.CopyRaster(
        in_raster=capa,
        out_rasterdataset=f"{out_gdb}\ortofoto_{year}_calasanz_copy",
        config_keyword="",
        background_value=None,
        nodata_value="",
        onebit_to_eightbit="NONE",
        colormap_to_RGB="NONE",
        pixel_type="",
        scale_pixel_value="NONE",
        RGB_to_Colormap="NONE",
        format="GRID",
        transform="NONE",
        process_as_multidimensional="CURRENT_SLICE",
        build_multidimensional_transpose="NO_TRANSPOSE"
    )
    print(f"Ráster {capa} copiado exitosamente")

<>:7: SyntaxWarning: invalid escape sequence '\o'
<>:12: SyntaxWarning: invalid escape sequence '\o'
C:/Users/USER/AppData/Local/Temp/ArcGISProTemp16508/xpython_16508/2888354055.py:7: SyntaxWarning: invalid escape sequence '\o'
  capa = f"Ortofotos\ortofoto_{year}_calasanz"
C:/Users/USER/AppData/Local/Temp/ArcGISProTemp16508/xpython_16508/2888354055.py:12: SyntaxWarning: invalid escape sequence '\o'
  out_rasterdataset=f"{out_gdb}\ortofoto_{year}_calasanz_copy",


Copiando ráster Ortofotos\ortofoto_2010_calasanz
Ráster Ortofotos\ortofoto_2010_calasanz copiado exitosamente
Copiando ráster Ortofotos\ortofoto_2016_calasanz
Ráster Ortofotos\ortofoto_2016_calasanz copiado exitosamente
Copiando ráster Ortofotos\ortofoto_2019_calasanz
Ráster Ortofotos\ortofoto_2019_calasanz copiado exitosamente
